In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Optional
import os
from pathlib import Path
import re
import json

# Set publication-ready style
plt.style.use('default')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 1.2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.8,
    'legend.frameon': True,
    'legend.fancybox': False,
    'legend.shadow': False,
    'legend.framealpha': 0.9,
    'legend.edgecolor': 'black',
    'legend.borderpad': 0.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'lines.linewidth': 2,
    'lines.markersize': 6,
})

# Set elegant color palette
colors_elegant = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#7209B7', '#2F9B5B']
sns.set_palette(colors_elegant)

# Configuration
# api_key = ""  # Fill in your wandb API key
wandb_project = "SAFE-long1k"
wandb_entity = "<WB ENTITY>"  # Leave None to use default entity

# Create output directory for PDFs
output_dir = Path("./paper_figures")
output_dir.mkdir(exist_ok=True)
print(f"Figures will be saved to: {output_dir.absolute()}")

Figures will be saved to: /Users/ihounie/repos/everysample-plots/paper_figures


In [113]:
api = wandb.Api()
# Get all runs from the project
runs = api.runs(f"{wandb_entity}/{wandb_project}" if wandb_entity else wandb_project)
# keep runs only with tag "10_ep" and config.num_train_epochs == 10
#runs = [r for r in runs if 'resilientt' in r.tags]
runs = [r for r in runs if 'cvar' in r.tags or 'resilient-cdf-5' in r.tags or 'resilient-cdf-10' in r.tags or 'resilient-cdf-20' in r.tags or 'dpo_5' in r.tags or 'other_avg' in r.tags]# and 'baseline_beavertails' in r.tags]
#runs = [r for r in runs if 'dual_1000' in r.tags or 'penalty_10' in r.tags or 'dual_1000_05' in r.tags or 'dual_100' in r.tags]
# exclde loss_type penalty
#runs = [r for r in runs if r.config.get('exp', {}).get('loss_type') not in ['penalty']]
print(f"Total runs found: {len(runs)}")

Total runs found: 9


In [114]:
def _logged_splits_with_coverage(runs, min_runs=1):
    """Return (splits, coverage) where coverage[split] = number of runs containing that split."""
    if not runs:
        return [], {}

    coverage = {}
    for run in runs:
        split_to_files = _constraint_files_by_split(run)
        for split, files in split_to_files.items():
            if files:
                coverage[split] = coverage.get(split, 0) + 1

    selected_splits = sorted(split for split, n in coverage.items() if n >= int(min_runs))
    return selected_splits, coverage
    
def _constraint_files_by_step(run, split):
    """Return {step: file_obj} for a given run/split, picking the latest epoch per step."""
    best_by_step = {}
    for f in run.files():
        m = pattern.match(f.name)
        if not m:
            continue
        epoch = float(m.group(1))
        file_split = m.group(2)
        step = int(m.group(3))

        # Historical quirk in some logs
        if step == 804:
            file_split = 'train'

        if file_split != split:
            continue

        prev = best_by_step.get(step)
        if prev is None:
            best_by_step[step] = (epoch, f)
        else:
            prev_epoch, prev_f = prev
            if (epoch, f.name) > (prev_epoch, prev_f.name):
                best_by_step[step] = (epoch, f)

    return {step: pair[1] for step, pair in best_by_step.items()}


CONSTRAINT_SLACKS_PATTERN = re.compile(
    r'^(?:.*/)?constraint_slacks_epoch_([0-9.]+)(?:_(train|eval))?_(\d+)_([A-Za-z0-9]+)\.table\.json$'
)

def _constraint_files_by_split(run):
    """Return {split: [file_obj, ...]} for constraint slacks, keeping one file per step."""
    best_by_split_and_step = {}
    for f in run.files():
        file_name = f.name or ''
        m = CONSTRAINT_SLACKS_PATTERN.match(file_name)
        if not m:
            continue

        epoch = float(m.group(1))
        split = (m.group(2) or 'unknown').lower()
        step = int(m.group(3))

        key = (split, step)
        prev = best_by_split_and_step.get(key)
        if prev is None:
            best_by_split_and_step[key] = (epoch, f)
        else:
            prev_epoch, prev_f = prev
            if (epoch, f.name) > (prev_epoch, prev_f.name):
                best_by_split_and_step[key] = (epoch, f)

    out = {}
    for (split, _step), (_, f) in best_by_split_and_step.items():
        out.setdefault(split, []).append(f)

    for split in out:
        out[split] = sorted(out[split], key=lambda x: x.name)

    return out


In [ ]:
# =========================================================================
# CDF subplots (Safe/Unsafe, Train/Test) mirroring the when2call notebook
# (plotting_preferences_when2call_simpo.ipynb ->
#  constraint_violation_loose_win_avg_log_likelihhod_*_all_train_test.pdf)
# =========================================================================

import colorsys
from matplotlib.lines import Line2D

# --- Style constants (match plotting_preferences_when2call_simpo.ipynb) ---
CDF_FIGSIZE = (6.8, 5.2)
CDF_TRAIN_TEST_FIGSIZE = (14.0, 6.4)
CDF_PLOT_LABEL_FONTSIZE = 26
CDF_PLOT_TICK_FONTSIZE = 21
CDF_PLOT_LEGEND_FONTSIZE = 21
CDF_PLOT_TITLE_FONTSIZE = 27
CDF_PLOT_LINE_WIDTH = 2.75

# Colors per method (match when2call + reranker notebooks).
CDF_METHOD_COLOR = {
    'base': '#6C757D',
    'avg': '#2E86AB',
    'point': '#A23B72',
    'dpo': '#F18F01',
    'res': '#C73E1D',
    'pen': '#2F9B5B',
    'simpo': '#7209B7',
}

# Safe/Unsafe encoded via linestyle (analogous to loose/win in the reference plot).
_CDF_SAFETY_LINESTYLE = {True: '-', False: '--'}  # True=Unsafe (solid), False=Safe (dashed)
_CDF_SAFETY_DISPLAY = {True: 'Unsafe', False: 'Safe'}


def _cdf_line_alpha_by_tag(subset_specs, lo=0.4, hi=1.0):
    """Map optional per-spec ``cdf_param`` (e.g. Δ or β) to line alpha, affine in log(param)."""
    params = {s['tag']: float(s['cdf_param']) for s in subset_specs if s.get('cdf_param') is not None}
    if not params:
        return {}
    tags = list(params.keys())
    logp = np.log(np.array([params[t] for t in tags], dtype=float))
    mn, mx = float(logp.min()), float(logp.max())
    out = {}
    if mx <= mn:
        a = (float(lo) + float(hi)) / 2.0
        for t in tags:
            out[t] = a
    else:
        span = float(hi) - float(lo)
        for t, lp in zip(tags, logp):
            tnorm = (lp - mn) / (mx - mn)
            out[t] = float(lo) + tnorm * span
    return out


def _shade_hex_color(hex_color, factor=0.6):
    """Darken (<1) or lighten (>1) an sRGB hex color by scaling HLS lightness."""
    s = hex_color.lstrip('#')
    r, g, b = (int(s[i:i + 2], 16) / 255.0 for i in (0, 2, 4))
    h, l, sat = colorsys.rgb_to_hls(r, g, b)
    l = max(0.0, min(1.0, l * float(factor)))
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, sat)
    return '#{:02X}{:02X}{:02X}'.format(int(round(r2 * 255)), int(round(g2 * 255)), int(round(b2 * 255)))


def _cdf_load_constraint_slacks(file_obj, run_id):
    """Self-contained loader that returns {True: [unsafe_slacks], False: [safe_slacks]}.

    Defined locally because a later cell (cell 5) shadows the original
    `_load_constraint_slacks` with a version that returns a plain list, which
    breaks the safety-label-aware `_collect_slacks_by_split_and_label` pipeline.
    """
    run_dir = Path('./wandb_downloads') / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    local_path = run_dir / file_obj.name
    file_obj.download(root=str(run_dir), replace=True)

    with open(local_path, 'r') as f:
        data = json.load(f)

    def _to_float_or_none(v):
        try:
            return float(v)
        except Exception:
            return None

    def _to_bool_or_none(v):
        if isinstance(v, bool):
            return v
        if isinstance(v, (int, np.integer)):
            return bool(v) if v in (0, 1) else None
        if isinstance(v, str):
            s = v.strip().lower()
            if s in ('true', '1', 'yes', 'y', 't'):
                return True
            if s in ('false', '0', 'no', 'n', 'f'):
                return False
        return None

    columns = data.get('columns', []) or []
    rows = data.get('data', []) or []

    preferred_col_names = ['constraint_slack', 'constraint_slacks', 'slack', 'value', 'mean_nll', 'nll']
    preferred_idx = None
    safety_idx = None
    if columns:
        lower_cols = [str(c).strip().lower() for c in columns]
        for name in preferred_col_names:
            if name in lower_cols:
                preferred_idx = lower_cols.index(name)
                break
        if 'safety_label' in lower_cols:
            safety_idx = lower_cols.index('safety_label')
    table_has_safety_col = safety_idx is not None

    slacks_by_label = {True: [], False: []}
    for row in rows:
        if row is None:
            continue
        raw_safety_label = None
        picked = None
        if isinstance(row, dict):
            for k, v in row.items():
                if str(k).strip().lower() == 'safety_label':
                    raw_safety_label = v
                    break
            if preferred_idx is not None and preferred_idx < len(columns):
                picked = _to_float_or_none(row.get(columns[preferred_idx]))
            if picked is None:
                for k, v in row.items():
                    if str(k).strip().lower() == 'safety_label':
                        continue
                    fv = _to_float_or_none(v)
                    if fv is not None:
                        picked = fv
                        break
        elif isinstance(row, (list, tuple)):
            if safety_idx is not None and safety_idx < len(row):
                raw_safety_label = row[safety_idx]
            if preferred_idx is not None and preferred_idx < len(row):
                picked = _to_float_or_none(row[preferred_idx])
            if picked is None:
                for j, v in enumerate(row):
                    if safety_idx is not None and j == safety_idx:
                        continue
                    fv = _to_float_or_none(v)
                    if fv is not None:
                        picked = fv
                        break
        else:
            picked = _to_float_or_none(row)

        safety_label = _to_bool_or_none(raw_safety_label)
        if picked is None:
            continue
        if safety_label not in (True, False):
            if table_has_safety_col:
                continue
            # Tables without a safety_label column (constraint_slack only) are treated
            # as unsafe by the upstream convention used elsewhere in this notebook.
            safety_label = True
        if safety_label is False:
            picked = -picked
        slacks_by_label[safety_label].append(picked)

    return slacks_by_label


def _cdf_collect_slacks_by_split_and_label(runs, splits):
    """Self-contained collector that uses `_cdf_load_constraint_slacks`."""
    out = {label: {split: [] for split in splits} for label in (True, False)}
    for run in runs:
        files_by_split = _constraint_files_by_split(run)
        for split in splits:
            files = files_by_split.get(split, [])
            if not files:
                continue
            merged_by_label = {True: [], False: []}
            for f in files:
                try:
                    loaded = _cdf_load_constraint_slacks(f, run.id)
                    for label in merged_by_label:
                        merged_by_label[label].extend(loaded.get(label, []))
                except Exception as e:
                    print(f'Failed to load slacks for {run.name} at split={split}, file={f.name}: {e}')
            for label, merged in merged_by_label.items():
                if not merged:
                    continue
                out[label][split].append({
                    'run_id': run.id,
                    'run_name': run.name,
                    'loss_type': run.config.get('exp', {}).get('loss_type'),
                    'slacks': np.array(merged, dtype=float),
                    'alpha': run.config.get('exp', {}).get('loss_alpha'),
                    'tol': run.config.get('exp', {}).get('tol'),
                    'epsilon': run.config.get('exp', {}).get('tol2'),
                    'split': split,
                    'safety_label': label,
                })
    return out


def _cdf_resolve_run_by_tag(project_runs, tag):
    matches = [r for r in project_runs if tag in (getattr(r, 'tags', None) or [])]
    if not matches:
        return None
    if len(matches) > 1:
        # Prefer the most recently created run for deterministic choice.
        matches = sorted(matches, key=lambda r: getattr(r, 'created_at', '') or '', reverse=True)
        print(f"Warning: {len(matches)} runs with tag={tag!r}; using {matches[0].name}")
    return matches[0]


def _plot_cdf_safety_train_test(
    subset_specs,
    figure_slug,
    all_project_runs,
    x_limits=None,
    y_limits=None,
    output_subdir='appendix/safe/cdf',
    x_scale='linear',
    symlog_linthresh=0.1,
):
    """Save one CDF figure (Train/Test panels) for a fixed subset of methods.

    subset_specs: list of dicts with keys {'tag','color','label'}; optional ``cdf_param`` (Δ, β, …)
    sets line alpha affine in log(cdf_param) across specs that define it; others use alpha=1.
    Safe uses dashed lines, Unsafe solid lines.
    Slacks are converted to "log likelihood": Unsafe stored=mean_nll -> negated; Safe stored=-mean_nll -> kept.
    """
    out_dir = Path(output_subdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    alpha_by_tag = _cdf_line_alpha_by_tag(subset_specs, lo=0.4, hi=1.0)
    resolved = []
    for spec in subset_specs:
        run = _cdf_resolve_run_by_tag(all_project_runs, spec['tag'])
        if run is None:
            print(f"[{figure_slug}] Skip tag={spec['tag']!r}: no matching run found")
            continue
        resolved.append({
            'run': run,
            'color': spec['color'],
            'label': spec['label'],
            'tag': spec['tag'],
            'line_alpha': alpha_by_tag.get(spec['tag'], 1.0),
        })

    if not resolved:
        print(f'[{figure_slug}] No runs resolved; abort.')
        return

    runs_subset = [r['run'] for r in resolved]
    splits, _cov = _logged_splits_with_coverage(runs_subset, min_runs=1)
    collected = _cdf_collect_slacks_by_split_and_label(runs_subset, splits)
    runid_to_spec = {r['run'].id: r for r in resolved}

    split_panel_order = [('train', 'Train'), ('eval', 'Test')]
    panels = []
    for internal, display in split_panel_order:
        entries = []
        for safety_label in (True, False):
            for entry in collected.get(safety_label, {}).get(internal, []):
                entries.append((safety_label, entry))
        if entries:
            panels.append((internal, display, entries))

    if not panels:
        print(f'[{figure_slug}] No train/test slack data available; skip.')
        return

    n_panels = len(panels)
    fig_w, fig_h = CDF_TRAIN_TEST_FIGSIZE if n_panels > 1 else CDF_FIGSIZE
    fig, axes = plt.subplots(1, n_panels, figsize=(fig_w, fig_h), sharey=True)
    if n_panels == 1:
        axes = [axes]
    else:
        axes = list(axes)

    bins = 1000
    resolved_order = {r['run'].id: i for i, r in enumerate(resolved)}

    for ax, (internal, display_name, entries) in zip(axes, panels):
        entries_sorted = sorted(
            entries,
            key=lambda es: (
                resolved_order.get(es[1]['run_id'], 10_000),
                0 if es[0] is True else 1,  # Unsafe first so solid is drawn beneath dashed highlights
            ),
        )
        for safety_label, entry in entries_sorted:
            spec = runid_to_spec.get(entry['run_id'])
            if spec is None:
                continue
            # Stored convention (from _cdf_load_constraint_slacks):
            #   unsafe -> +mean_nll,   safe -> -mean_nll (already negated in loader).
            # We want both plotted as -|stored| so the x-axis is always the signed slack
            # with safe values appearing negative alongside unsafe. Negate both labels.
            slacks = -np.asarray(entry['slacks'], dtype=float)
            # Apply per-label offsets to align curves on a common scale.
            if safety_label is False:
                slacks = slacks - 2.0
            else:  # unsafe
                slacks = slacks - 0.04
            if x_scale == 'symlog':
                slacks = slacks #- 0.000001
            counts, edges = np.histogram(slacks, bins=bins)
            total = counts.sum()
            cdf = np.cumsum(counts) / total if total > 0 else np.zeros_like(counts, dtype=float)
            ax.plot(
                edges[1:],
                cdf,
                linewidth=CDF_PLOT_LINE_WIDTH,
                linestyle=_CDF_SAFETY_LINESTYLE[safety_label],
                color=spec['color'],
                alpha=spec['line_alpha'],
                label='_nolegend_',
            )

        ax.set_title(display_name, fontsize=CDF_PLOT_TITLE_FONTSIZE)
        ax.tick_params(axis='y', labelsize=CDF_PLOT_TICK_FONTSIZE)
        ax.tick_params(axis='x', labelsize=CDF_PLOT_TICK_FONTSIZE, pad=2.5)
        ax.set_ylim(*(y_limits if y_limits is not None else (-0.025, 1.1)))
        if x_scale == 'symlog':
            # symmetric log: linear near 0 (|x| < linthresh), log further out;
            # needed because the slack range includes 0 and negative values.
            ax.set_xscale('symlog', linthresh=symlog_linthresh)
        elif x_scale and x_scale != 'linear':
            ax.set_xscale(x_scale)
        if x_limits is not None:
            _xl, _xr = x_limits
            _xlim_kw = {}
            if _xl is not None:
                _xlim_kw['left'] = _xl
            if _xr is not None:
                _xlim_kw['right'] = _xr
            if _xlim_kw:
                ax.set_xlim(**_xlim_kw)
        ax.grid(True, axis='y', alpha=0.3, linewidth=0.8)
        ax.grid(False, axis='x')
        ax.set_axisbelow(True)

    axes[0].set_ylabel('Cumulative Density', fontsize=CDF_PLOT_LABEL_FONTSIZE)

    legend_line_width = max(2.0, CDF_PLOT_LINE_WIDTH)
    constraint_header = Line2D([0], [0], linestyle='None', linewidth=0.0, color='none', label='Constraint')
    method_header = Line2D([0], [0], linestyle='None', linewidth=0.0, color='none', label='Method')
    style_handles = [
        Line2D([0], [0], color='black', linestyle=_CDF_SAFETY_LINESTYLE[True],
               linewidth=legend_line_width, label=_CDF_SAFETY_DISPLAY[True]),
        Line2D([0], [0], color='black', linestyle=_CDF_SAFETY_LINESTYLE[False],
               linewidth=legend_line_width, label=_CDF_SAFETY_DISPLAY[False]),
    ]
    method_handles = [
        Line2D([0], [0], color=r['color'], linestyle='-',
               linewidth=legend_line_width, alpha=r['line_alpha'], label=r['label'])
        for r in resolved
    ]
    legend_handles = [constraint_header] + style_handles + [method_header] + method_handles
    legend_labels = [h.get_label() for h in legend_handles]
    leg = fig.legend(
        legend_handles,
        legend_labels,
        loc='center left',
        bbox_to_anchor=(0.905, 0.5),
        frameon=False,
        fontsize=CDF_PLOT_LEGEND_FONTSIZE,
        handlelength=2.0,
        borderaxespad=0.0,
    )
    legend_texts = leg.get_texts()
    if legend_texts:
        legend_texts[0].set_ha('center')
        legend_texts[0].set_fontweight('bold')
    if len(legend_texts) > len(style_handles) + 1:
        legend_texts[len(style_handles) + 1].set_ha('center')
        legend_texts[len(style_handles) + 1].set_fontweight('bold')

    plt.tight_layout(rect=[0.04, 0.055, 0.90, 0.96])
    fig.supxlabel('Average log likelihood', fontsize=CDF_PLOT_LABEL_FONTSIZE, y=0.028)

    filename = f"constraint_violation_loglik_safety_{figure_slug}_train_test.pdf"
    filepath = out_dir / filename
    plt.savefig(filepath)
    print(f'Saved figure to {filepath}')
    plt.show()


# --- Fetch runs broadly: baseline_beavertails may not carry the 'both' tag. ---
_CDF_RELEVANT_TAGS = {
    'baseline_beavertails',
    'avg_both',
    'penalty_both',
    'point', 'dpo_10', 'dpo_5', 'dpo_2', 'resilient-cdf-5',# 'resilient-cdf-10', 'resilient-cdf-20',
    'resilientt', 'resilientttt',
}
_cdf_project_runs = [
    r for r in api.runs(f"{wandb_entity}/{wandb_project}" if wandb_entity else wandb_project)
    if set(getattr(r, 'tags', None) or []) & _CDF_RELEVANT_TAGS
]
print(f'CDF-relevant runs fetched: {len(_cdf_project_runs)}')
for _r in _cdf_project_runs:
    print(f"  {_r.name}  tags={list(_r.tags or [])}")

# --- Distinct colors for within-method variants. ---
# Chosen so each variant is easy to distinguish from its partner AND from
# 'point' (#A23B72, magenta), which is present in every plot.
_dpo_primary   = CDF_METHOD_COLOR['dpo']   # '#F18F01' orange
_dpo_secondary = '#B8860B'                 # dark goldenrod – clearly warm/yellow, far from magenta
_res_primary   = CDF_METHOD_COLOR['res']   # '#C73E1D' red
_res_secondary = '#2E86AB'                 # blue – clearly distinct from red (β=10) and magenta (point)

_cdf_subsets = [
    {
        'slug': 'avg_pen_point',
        'specs': [
            {'tag': 'avg_both',             'color': CDF_METHOD_COLOR['avg'],   'label':r'avg ($\epsilon=-0.28$)' },
            {'tag': 'penalty_both',         'color': CDF_METHOD_COLOR['pen'],   'label': 'pen ($\lambda_0$=5)'},
            {'tag': 'dpo_10',               'color': _dpo_secondary,            'label': r'DPO ($\Delta=10$)'},
            {'tag': 'point',                'color': CDF_METHOD_COLOR['point'], 'label': 'point'},
        ],
        # Symlog x-axis only for this plot, right edge clamped at 0.1.
        'x_scale': 'symlog',
        'x_limits': (None, -0.001),
    },
    {
        'slug': 'res_point',
        'specs': [
            {'tag': 'resilientt',           'color': _res_primary,              'label': r'relax ($\beta=10$)', 'cdf_param': 10},
            {'tag': 'resilientttt',           'color': _res_primary,              'label': r'relax ($\beta=20$)', 'cdf_param': 20},
            {'tag': 'resilient-cdf-5',           'color': _res_primary,              'label': r'relax ($\beta=5$)', 'cdf_param': 5},
            {'tag': 'point',                'color': CDF_METHOD_COLOR['point'], 'label': 'point'},
        ],
    },
]

for _sub in _cdf_subsets:
    _plot_cdf_safety_train_test(
        _sub['specs'],
        figure_slug=_sub['slug'],
        all_project_runs=_cdf_project_runs,
        x_scale=_sub.get('x_scale', 'linear'),
        x_limits=_sub.get('x_limits'),
    )


CDF-relevant runs fetched: 11
  run-alpha:1000.0-tol:-0.04-lr:1.0  tags=['agd', 'both', 'dual_both_2', 'pareto', 'point', 'sample', 'xtest']
  run-alpha:5.0-tol:0.0-lr:0  tags=['both', 'cvar', 'final', 'pareto', 'penalty_both', 'sample', 'tab2', 'xtest']
  run-alpha:1000.0-tol:-0.04-lr:1.0  tags=['both', 'kl', 'pku', 'resilientt', 'sample', 'xtest2']
  run-alpha:1000.0-tol:-0.04-lr:1.0  tags=['resilientttt', 'xtest2']
  run-alpha:1000.0-tol:-0.04-lr:1.0  tags=['both', 'kl', 'pku', 'resilientttt', 'sample', 'xtest2']
  run-alpha:0.0-tol:0.0-lr:0  tags=['baseline_beavertails', 'both', 'final', 'sample', 'xtest']
  run-daa-alpha:0.1-eps:10.0-only_unsafe:false  tags=['both', 'daa', 'dpo', 'dpo_10', 'final', 'pareto', 'ref', 'sample']
  run-daa-alpha:0.1-eps:5.0-only_unsafe:false  tags=['both', 'daa', 'dpo_5', 'final', 'pareto', 'ref', 'sample']
  run-daa-alpha:0.1-eps:2.0-only_unsafe:false  tags=['both', 'daa', 'dpo_2', 'final', 'pareto', 'ref', 'sample']
  run-alpha:1000.0-tol:-0.28-lr:1.